***5B) PER-PATIENT PSEUDOBULK INTERACTION SCORING***

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "03B_GSE135779_DONOR_LIANA_SCORING")


In [ ]:
import pandas as pd
from pathlib import Path

def extract_per_patient_liana_scores(liana_res_path, out_path):
    """Create one LIANA consensus score per donor and interaction.

    LIANA must have been run with ``rank_aggregate.by_sample``. Consensus
    magnitude rank is converted to an intuitive score where larger is stronger.
    No custom ligand/receptor expression product is calculated.
    """
    result = pd.read_csv(liana_res_path)
    required = {"sample", "source", "target", "ligand_complex",
                "receptor_complex", "magnitude_rank"}
    missing = required.difference(result.columns)
    if missing:
        raise ValueError(
            f"{liana_res_path} is not a by-sample LIANA result; missing {sorted(missing)}"
        )
    result["interaction_id"] = (
        result["source"].astype(str) + "|" + result["target"].astype(str) + "|" +
        result["ligand_complex"].astype(str) + "|" + result["receptor_complex"].astype(str)
    )
    result["score"] = 1.0 - result["magnitude_rank"].astype(float)
    keep = ["sample", "interaction_id", "source", "target", "ligand_complex",
            "receptor_complex", "score", "magnitude_rank"]
    if result.duplicated(["sample", "interaction_id"]).any():
        raise ValueError("LIANA returned duplicate donor-interaction rows.")
    result = result[keep].sort_values(["interaction_id", "sample"])
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(out_path, index=False)
    print(f"Saved {len(result)} donor-interaction scores to {out_path}")
    print(f"Donors scored: {result['sample'].nunique()}")
    return result


In [ ]:
print("=== CHILD (cSLE) per-patient scoring ===")
child_scores = extract_per_patient_liana_scores(
    liana_res_path=f"{BASE_DIR}/Results/cSLE_liana/cSLE_liana_res.csv",
    out_path=f"{BASE_DIR}/Results/severity_analysis/cSLE_per_patient_scores.csv",
)
child_scores.head()

In [ ]:
print("=== ADULT (aSLE) per-patient scoring ===")
adult_scores = extract_per_patient_liana_scores(
    liana_res_path=f"{BASE_DIR}/Results/aSLE_liana/aSLE_liana_res.csv",
    out_path=f"{BASE_DIR}/Results/severity_analysis/aSLE_per_patient_scores.csv",
)
adult_scores.head()

**Extending to the healthy cohorts.** The severity analysis (06A/06B) only needed donor-level scores for the SLE cohorts (cSLE/aSLE), since SLEDAI is only measured in SLE patients. The group-comparison statistics rebuild (04A-04F and 05) needs donor-level scores for the healthy cohorts too, to run a real two-group test instead of the old single-pooled-score rank difference.

In [ ]:
print("=== CHILD (cH) per-patient scoring ===")
ch_scores = extract_per_patient_liana_scores(
    liana_res_path=f"{BASE_DIR}/Results/cH_liana/cH_liana_res.csv",
    out_path=f"{BASE_DIR}/Results/severity_analysis/cH_per_patient_scores.csv",
)
ch_scores.head()

In [ ]:
print("=== ADULT (aH) per-patient scoring ===")
ah_scores = extract_per_patient_liana_scores(
    liana_res_path=f"{BASE_DIR}/Results/aH_liana/aH_liana_res.csv",
    out_path=f"{BASE_DIR}/Results/severity_analysis/aH_per_patient_scores.csv",
)
ah_scores.head()

**Per-patient interaction score distribution (sanity check).**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, df, label in [(axes[0], child_scores, "cSLE"), (axes[1], adult_scores, "aSLE")]:
    nonzero = df["score"]
    ax.hist(nonzero, bins=50, color="#5B3A73", alpha=0.8)
    ax.set_xlabel("LIANA consensus communication score (1 - magnitude rank)")
    ax.set_ylabel("Count")
    ax.set_title(f"{label}: per-patient interaction scores (n={df['sample'].nunique()} patients)")
plt.tight_layout()
out_path = f"{BASE_DIR}/Results/severity_analysis/perpatient_score_distributions.png"
plt.savefig(out_path, dpi=600)
plt.show()
print("Saved:", out_path)